In [0]:
# Cell: Green Taxi Silver Transformation
from pyspark.sql.functions import (
    col, when, unix_timestamp, round as spark_round,
    year, month, dayofweek, hour, current_timestamp
)

print("Loading Green Taxi Bronze...")
green_silver = spark.read.table("jeit_kg_dev.default.green_taxi_bronze")

print(f"Bronze records: {green_silver.count():,}")

# ============================================
# STEP 1: DATA CLEANING & VALIDATION
# ============================================

print("\n=== STEP 1: Data Cleaning ===")

# Remove null values in critical fields
green_silver = green_silver.dropna(subset=[
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "trip_distance",
    "total_amount"
])

# Filter invalid dates (pickup must be before dropoff)
green_silver = green_silver.filter(
    (col("lpep_pickup_datetime") < col("lpep_dropoff_datetime")) &
    (col("lpep_pickup_datetime") <= current_timestamp())
)

# Remove duplicates
green_silver = green_silver.dropDuplicates([
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime", 
    "PULocationID",
    "DOLocationID",
    "total_amount"
])

print(f"After cleaning: {green_silver.count():,} records")

# ============================================
# STEP 2: CALCULATED COLUMNS
# ============================================

print("\n=== STEP 2: Adding Calculated Columns ===")

# Trip duration in minutes
green_silver = green_silver.withColumn(
    "trip_duration_minutes",
    spark_round(
        (unix_timestamp("lpep_dropoff_datetime") - unix_timestamp("lpep_pickup_datetime")) / 60,
        2
    )
)

# Trip efficiency (miles per minute)
green_silver = green_silver.withColumn(
    "trip_efficiency",
    spark_round(
        col("trip_distance") / col("trip_duration_minutes"),
        4
    )
)

# Fare per mile
green_silver = green_silver.withColumn(
    "fare_per_mile",
    spark_round(
        col("fare_amount") / col("trip_distance"),
        2
    )
)

# ============================================
# STEP 3: OUTLIER FILTERING
# ============================================

print("\n=== STEP 3: Filtering Outliers ===")

green_silver = green_silver.filter(
    # Passenger count: 1-6
    (col("passenger_count") >= 1) & (col("passenger_count") <= 6) &
    
    # Trip distance: 0.1-100 miles
    (col("trip_distance") > 0.1) & (col("trip_distance") <= 100) &
    
    # Trip duration: 1-180 minutes
    (col("trip_duration_minutes") >= 1) & (col("trip_duration_minutes") <= 180) &
    
    # Fare amount: positive and reasonable
    (col("fare_amount") > 0) & (col("fare_amount") <= 500) &
    
    # Total amount: positive and reasonable
    (col("total_amount") > 0) & (col("total_amount") <= 500) &
    
    # Trip efficiency: reasonable (0.01 to 2 miles/min = 0.6 to 120 mph)
    (col("trip_efficiency") > 0.01) & (col("trip_efficiency") < 2)
)

print(f"After outlier filtering: {green_silver.count():,} records")

# ============================================
# STEP 4: DECODED COLUMNS
# ============================================

print("\n=== STEP 4: Adding Decoded Columns ===")

# Decode Rate Code
green_silver = green_silver.withColumn("rate_code_desc",
    when(col("RatecodeID") == 1, "Standard rate")
    .when(col("RatecodeID") == 2, "JFK")
    .when(col("RatecodeID") == 3, "Newark")
    .when(col("RatecodeID") == 4, "Nassau or Westchester")
    .when(col("RatecodeID") == 5, "Negotiated fare")
    .when(col("RatecodeID") == 6, "Group ride")
    .when(col("RatecodeID") == 99, "Null/unknown")
    .otherwise("Unknown")
)

# Decode Payment Type
green_silver = green_silver.withColumn("payment_type_desc",
    when(col("payment_type") == 0, "Flex Fare trip")
    .when(col("payment_type") == 1, "Credit card")
    .when(col("payment_type") == 2, "Cash")
    .when(col("payment_type") == 3, "No charge")
    .when(col("payment_type") == 4, "Dispute")
    .when(col("payment_type") == 5, "Unknown")
    .when(col("payment_type") == 6, "Voided trip")
    .otherwise("Unknown")
)

# Decode Trip Type
green_silver = green_silver.withColumn("trip_type_desc",
    when(col("trip_type") == 1, "Street-hail")
    .when(col("trip_type") == 2, "Dispatch")
    .otherwise("Unknown")
)

# Decode Store and Forward Flag
green_silver = green_silver.withColumn("store_and_fwd_flag_desc",
    when(col("store_and_fwd_flag") == "Y", "Store and forward trip")
    .when(col("store_and_fwd_flag") == "N", "Not a store and forward trip")
    .otherwise("Unknown")
)

# ============================================
# STEP 5: TIME-BASED COLUMNS
# ============================================

print("\n=== STEP 5: Adding Time-Based Columns ===")

green_silver = green_silver \
    .withColumn("pickup_hour", hour("lpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", dayofweek("lpep_pickup_datetime")) \
    .withColumn("pickup_year", year("lpep_pickup_datetime")) \
    .withColumn("pickup_month", month("lpep_pickup_datetime"))

# Day of week description
green_silver = green_silver.withColumn("day_of_week_desc",
    when(col("pickup_day_of_week") == 1, "Sunday")
    .when(col("pickup_day_of_week") == 2, "Monday")
    .when(col("pickup_day_of_week") == 3, "Tuesday")
    .when(col("pickup_day_of_week") == 4, "Wednesday")
    .when(col("pickup_day_of_week") == 5, "Thursday")
    .when(col("pickup_day_of_week") == 6, "Friday")
    .when(col("pickup_day_of_week") == 7, "Saturday")
    .otherwise("Unknown")
)

# Time of day classification
green_silver = green_silver.withColumn("time_of_day",
    when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), "Morning")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), "Afternoon")
    .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), "Evening")
    .otherwise("Night")
)

# ============================================
# STEP 6: SAVE TO SILVER
# ============================================

print("\n=== STEP 6: Saving to Silver Layer ===")

green_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("pickup_year", "pickup_month") \
    .saveAsTable("jeit_kg_dev.default.green_taxi_silver")

print("✅ green_taxi_silver created!")

Loading Green Taxi Bronze...
Bronze records: 1,990,417

=== STEP 1: Data Cleaning ===
After cleaning: 1,986,848 records

=== STEP 2: Adding Calculated Columns ===

=== STEP 3: Filtering Outliers ===
After outlier filtering: 1,716,600 records

=== STEP 4: Adding Decoded Columns ===

=== STEP 5: Adding Time-Based Columns ===

=== STEP 6: Saving to Silver Layer ===
✅ green_taxi_silver created!


In [0]:

# ============================================
# STEP 7: DATA QUALITY REPORT
# ============================================

print("\n" + "="*60)
print("       GREEN TAXI SILVER - QUALITY REPORT")
print("="*60)

final_count = spark.sql("SELECT COUNT(*) FROM jeit_kg_dev.default.green_taxi_silver").collect()[0][0]
print(f"\n✅ Final record count: {final_count:,}")
print(f"   Records removed:    {green_silver.count():,}")
print(f"   Removal rate:       {((1 - final_count/1990417) * 100):.1f}%")

print("\n📊 Quality Checks:")
spark.sql("""
    SELECT 
        'Null pickup times' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.green_taxi_silver
    WHERE lpep_pickup_datetime IS NULL
    
    UNION ALL
    
    SELECT 
        'Invalid distances' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.green_taxi_silver
    WHERE trip_distance <= 0 OR trip_distance > 100
    
    UNION ALL
    
    SELECT 
        'Invalid fares' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.green_taxi_silver
    WHERE total_amount <= 0
""").show()

print("\n📊 Sample Statistics:")
spark.sql("""
    SELECT 
        pickup_year,
        COUNT(*) as trips,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(trip_duration_minutes), 2) as avg_duration,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_efficiency), 4) as avg_efficiency
    FROM jeit_kg_dev.default.green_taxi_silver
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()


       GREEN TAXI SILVER - QUALITY REPORT

✅ Final record count: 1,716,600
   Records removed:    1,716,600
   Removal rate:       13.8%

📊 Quality Checks:
+-----------------+-----+
|       check_name|count|
+-----------------+-----+
|Null pickup times|    0|
|Invalid distances|    0|
|    Invalid fares|    0|
+-----------------+-----+


📊 Sample Statistics:
+-----------+------+------------+------------+--------+--------------+
|pickup_year| trips|avg_distance|avg_duration|avg_fare|avg_efficiency|
+-----------+------+------------+------------+--------+--------------+
|       2008|     1|        2.25|        20.0|   21.25|        0.1125|
|       2009|     3|        0.97|        6.45|   10.88|        0.1536|
|       2022|     2|        5.43|       24.51|    26.2|        0.3043|
|       2023|674345|        2.92|       14.47|   23.06|         0.196|
|       2024|582930|         2.9|       14.61|   23.76|        0.1928|
|       2025|459319|        2.96|       14.83|   24.64|        0.1924|

In [0]:
# Cell: Yellow Taxi Silver - Process Year by Year
from pyspark.sql.functions import (
    col, when, unix_timestamp, round as spark_round,
    year, month, dayofweek, hour, current_timestamp
)

# Configuration
table_name = "jeit_kg_dev.default.yellow_taxi_silver"
source_table = "jeit_kg_dev.default.yellow_taxi_bronze"

# Drop existing table if starting fresh
spark.sql(f"DROP TABLE IF EXISTS {table_name}")

# Reduce shuffle partitions for better memory management
spark.conf.set("spark.sql.shuffle.partitions", "100")

print("="*70)
print("     YELLOW TAXI SILVER TRANSFORMATION")
print("="*70)

# Process each year separately
for yr in [2023, 2024, 2025]:
    print(f"\n{'='*70}")
    print(f"Processing {yr}...")
    print('='*70)
    
    # Read only this year's data
    yellow_year = spark.sql(f"""
        SELECT * 
        FROM {source_table}
        WHERE year = {yr}
    """)
    
    initial_count = yellow_year.count()
    print(f"Initial records for {yr}: {initial_count:,}")
    
    # ============================================
    # STEP 1: DATA CLEANING & VALIDATION
    # ============================================
    
    print(f"\n  Step 1: Cleaning {yr} data...")
    
    yellow_year = yellow_year.dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "total_amount"
    ])
    
    yellow_year = yellow_year.filter(
        (col("tpep_pickup_datetime") < col("tpep_dropoff_datetime")) &
        (col("tpep_pickup_datetime") <= current_timestamp())
    )
    
    yellow_year = yellow_year.dropDuplicates([
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime", 
        "PULocationID",
        "DOLocationID",
        "total_amount"
    ])
    
    after_cleaning = yellow_year.count()
    print(f"  After cleaning: {after_cleaning:,} records ({initial_count - after_cleaning:,} removed)")
    
    # ============================================
    # STEP 2: CALCULATED COLUMNS
    # ============================================
    
    print(f"\n  Step 2: Adding calculated columns...")
    
    yellow_year = yellow_year.withColumn(
        "trip_duration_minutes",
        spark_round(
            (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60,
            2
        )
    )
    
    yellow_year = yellow_year.withColumn(
        "trip_efficiency",
        spark_round(
            col("trip_distance") / col("trip_duration_minutes"),
            4
        )
    )
    
    yellow_year = yellow_year.withColumn(
        "fare_per_mile",
        spark_round(
            col("fare_amount") / col("trip_distance"),
            2
        )
    )
    
    # ============================================
    # STEP 3: OUTLIER FILTERING
    # ============================================
    
    print(f"\n  Step 3: Filtering outliers...")
    
    yellow_year = yellow_year.filter(
        (col("passenger_count") >= 1) & (col("passenger_count") <= 6) &
        (col("trip_distance") > 0.1) & (col("trip_distance") <= 100) &
        (col("trip_duration_minutes") >= 1) & (col("trip_duration_minutes") <= 180) &
        (col("fare_amount") > 0) & (col("fare_amount") <= 500) &
        (col("total_amount") > 0) & (col("total_amount") <= 500) &
        (col("trip_efficiency") > 0.01) & (col("trip_efficiency") < 2)
    )
    
    after_outliers = yellow_year.count()
    print(f"  After outlier filtering: {after_outliers:,} records ({after_cleaning - after_outliers:,} removed)")
    
    # ============================================
    # STEP 4: DECODED COLUMNS
    # ============================================
    
    print(f"\n  Step 4: Adding decoded columns...")
    
    # Rate Code
    yellow_year = yellow_year.withColumn("rate_code_desc",
        when(col("RatecodeID") == 1, "Standard rate")
        .when(col("RatecodeID") == 2, "JFK")
        .when(col("RatecodeID") == 3, "Newark")
        .when(col("RatecodeID") == 4, "Nassau or Westchester")
        .when(col("RatecodeID") == 5, "Negotiated fare")
        .when(col("RatecodeID") == 6, "Group ride")
        .when(col("RatecodeID") == 99, "Null/unknown")
        .otherwise("Unknown")
    )
    
    # Payment Type
    yellow_year = yellow_year.withColumn("payment_type_desc",
        when(col("payment_type") == 0, "Flex Fare trip")
        .when(col("payment_type") == 1, "Credit card")
        .when(col("payment_type") == 2, "Cash")
        .when(col("payment_type") == 3, "No charge")
        .when(col("payment_type") == 4, "Dispute")
        .when(col("payment_type") == 5, "Unknown")
        .when(col("payment_type") == 6, "Voided trip")
        .otherwise("Unknown")
    )
    
    # Store and Forward Flag
    yellow_year = yellow_year.withColumn("store_and_fwd_flag_desc",
        when(col("store_and_fwd_flag") == "Y", "Store and forward trip")
        .when(col("store_and_fwd_flag") == "N", "Not a store and forward trip")
        .otherwise("Unknown")
    )
    
    # ============================================
    # STEP 5: TIME-BASED COLUMNS
    # ============================================
    
    print(f"\n  Step 5: Adding time-based columns...")
    
    yellow_year = yellow_year \
        .withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
        .withColumn("pickup_day_of_week", dayofweek("tpep_pickup_datetime")) \
        .withColumn("pickup_year", year("tpep_pickup_datetime")) \
        .withColumn("pickup_month", month("tpep_pickup_datetime"))
    
    # Day of week description
    yellow_year = yellow_year.withColumn("day_of_week_desc",
        when(col("pickup_day_of_week") == 1, "Sunday")
        .when(col("pickup_day_of_week") == 2, "Monday")
        .when(col("pickup_day_of_week") == 3, "Tuesday")
        .when(col("pickup_day_of_week") == 4, "Wednesday")
        .when(col("pickup_day_of_week") == 5, "Thursday")
        .when(col("pickup_day_of_week") == 6, "Friday")
        .when(col("pickup_day_of_week") == 7, "Saturday")
        .otherwise("Unknown")
    )
    
    # Time of day
    yellow_year = yellow_year.withColumn("time_of_day",
        when((col("pickup_hour") >= 6) & (col("pickup_hour") < 12), "Morning")
        .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), "Afternoon")
        .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), "Evening")
        .otherwise("Night")
    )
    
    # ============================================
    # STEP 6: SAVE THIS YEAR'S DATA
    # ============================================
    
    print(f"\n  Step 6: Saving {yr} to Silver...")
    
    yellow_year.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("pickup_year", "pickup_month") \
        .saveAsTable(table_name)
    
    print(f"✅ {yr} complete: {after_outliers:,} records saved")
    
    # Clear cache to free memory
    yellow_year.unpersist()

print("\n" + "="*70)
print("     YELLOW TAXI SILVER TRANSFORMATION COMPLETE")
print("="*70)

     YELLOW TAXI SILVER TRANSFORMATION

Processing 2023...
Initial records for 2023: 38,310,132

  Step 1: Cleaning 2023 data...
  After cleaning: 38,294,557 records (15,575 removed)

  Step 2: Adding calculated columns...

  Step 3: Filtering outliers...
  After outlier filtering: 35,416,849 records (2,877,708 removed)

  Step 4: Adding decoded columns...

  Step 5: Adding time-based columns...

  Step 6: Saving 2023 to Silver...
✅ 2023 complete: 35,416,849 records saved

Processing 2024...
Initial records for 2024: 41,169,691

  Step 1: Cleaning 2024 data...
  After cleaning: 41,156,172 records (13,519 removed)

  Step 2: Adding calculated columns...

  Step 3: Filtering outliers...
  After outlier filtering: 35,430,651 records (5,725,521 removed)

  Step 4: Adding decoded columns...

  Step 5: Adding time-based columns...

  Step 6: Saving 2024 to Silver...
✅ 2024 complete: 35,430,651 records saved

Processing 2025...
Initial records for 2025: 44,417,572

  Step 1: Cleaning 2025 dat

In [0]:
# Cell: Yellow Silver Quality Report
print("="*70)
print("      YELLOW TAXI SILVER - FINAL QUALITY REPORT")
print("="*70)

# Total count
total_count = spark.sql("SELECT COUNT(*) FROM jeit_kg_dev.default.yellow_taxi_silver").collect()[0][0]
print(f"\n✅ Total Silver Records: {total_count:,}")

# By year
print("\n📊 Records by Year:")
spark.sql("""
    SELECT 
        pickup_year,
        COUNT(*) as trips,
        ROUND(AVG(trip_distance), 2) as avg_distance,
        ROUND(AVG(trip_duration_minutes), 2) as avg_duration,
        ROUND(AVG(total_amount), 2) as avg_fare,
        ROUND(AVG(trip_efficiency), 4) as avg_efficiency
    FROM jeit_kg_dev.default.yellow_taxi_silver
    GROUP BY pickup_year
    ORDER BY pickup_year
""").show()

# Payment type distribution
print("\n📊 Payment Type Distribution:")
spark.sql("""
    SELECT 
        payment_type_desc,
        COUNT(*) as trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM jeit_kg_dev.default.yellow_taxi_silver
    GROUP BY payment_type_desc
    ORDER BY trips DESC
""").show()

# Rate code distribution
print("\n📊 Rate Code Distribution:")
spark.sql("""
    SELECT 
        rate_code_desc,
        COUNT(*) as trips,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as pct
    FROM jeit_kg_dev.default.yellow_taxi_silver
    GROUP BY rate_code_desc
    ORDER BY trips DESC
""").show()

# Quality checks
print("\n✅ Quality Checks (all should be 0):")
spark.sql("""
    SELECT 
        'Null pickup times' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.yellow_taxi_silver
    WHERE tpep_pickup_datetime IS NULL
    
    UNION ALL
    
    SELECT 
        'Invalid distances' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.yellow_taxi_silver
    WHERE trip_distance <= 0 OR trip_distance > 100
    
    UNION ALL
    
    SELECT 
        'Invalid durations' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.yellow_taxi_silver
    WHERE trip_duration_minutes <= 0 OR trip_duration_minutes > 180
    
    UNION ALL
    
    SELECT 
        'Invalid fares' as check_name,
        COUNT(*) as count
    FROM jeit_kg_dev.default.yellow_taxi_silver
    WHERE total_amount <= 0 OR total_amount > 500
""").show()

print("\n🎉 Yellow Taxi Silver layer complete!")

      YELLOW TAXI SILVER - FINAL QUALITY REPORT

✅ Total Silver Records: 102,796,467

📊 Records by Year:
+-----------+--------+------------+------------+--------+--------------+
|pickup_year|   trips|avg_distance|avg_duration|avg_fare|avg_efficiency|
+-----------+--------+------------+------------+--------+--------------+
|       2023|35416849|        3.52|       16.46|   28.89|        0.1898|
|       2024|35430651|        3.45|       16.95|   28.98|         0.181|
|       2025|31948967|        3.43|       16.88|   29.36|        0.1805|
+-----------+--------+------------+------------+--------+--------------+


📊 Payment Type Distribution:
+-----------------+--------+-----+
|payment_type_desc|   trips|  pct|
+-----------------+--------+-----+
|      Credit card|86351156|84.00|
|             Cash|15018448|14.61|
|          Dispute| 1050735| 1.02|
|        No charge|  376127| 0.37|
|          Unknown|       1| 0.00|
+-----------------+--------+-----+


📊 Rate Code Distribution:
+---------